# 09 — Pipeline de Retrieval (online)

Dado el perfil clinico de un paciente (formulario del frontend), recupera y rankea los
ensayos clinicos disponibles. Reusa los embeddings generados offline en el notebook 06.

**Flujo**
1. Formulario del paciente -> texto de perfil
2. Embedding del perfil (mismos modelos que offline)
3. Filtros duros: status RECRUITING + edad compatible
4. Filtro semantico de condicion -> estudios candidatos (top-N)
5. Match semantico contra criterios de **inclusion** (score positivo)
6. Penalizacion semantica por criterios de **exclusion**
7. Score compuesto -> top-N estudios + criterio que matchea (explicabilidad)

**Limitaciones de los datos actuales** (`05_clinical_trials_normalized.parquet`):
no hay columna de **sexo** ni de **pais/geografia**, y aun no hay cross-encoder.
Esos pasos del diseno quedan como hooks marcados `TODO`.

**Input**
- `data/05_clinical_trials_normalized.parquet` — filtros (status, edad) + condicion
- `data/06_{inclusion,exclusion,condition}_meta.parquet` + `.npy` — embeddings offline


## 1 — Setup

In [1]:
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

C:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path('../data')

# --- Datos de estudios (outputs del pipeline offline) ---
CLINICAL_PARQUET    = DATA_DIR / '05_clinical_trials_normalized.parquet'
INCLUSION_META_PATH = DATA_DIR / '06_inclusion_meta.parquet'
INCLUSION_EMB_PATH  = DATA_DIR / '06_inclusion_embeddings.npy'
EXCLUSION_META_PATH = DATA_DIR / '06_exclusion_meta.parquet'
EXCLUSION_EMB_PATH  = DATA_DIR / '06_exclusion_embeddings.npy'
CONDITION_META_PATH = DATA_DIR / '06_condition_meta.parquet'
CONDITION_EMB_PATH  = DATA_DIR / '06_condition_embeddings.npy'

# --- Modelos (mismos que el pipeline offline, notebook 06) ---
CRITERIA_MODEL_NAME  = 'pritamdeka/S-PubMedBert-MS-MARCO'  # 768 dims -> inclusion/exclusion
CONDITION_MODEL_NAME = 'BAAI/bge-small-en-v1.5'            # 384 dims -> filtro de condicion

# --- Filtros y scoring ---
RECRUITING_STATUS = 'RECRUITING'
CONDITION_TOP_N   = 200   # estudios candidatos tras el filtro semantico de condicion
INCLUSION_TOP_K   = 3     # criterios promediados por estudio para el score
RESULT_TOP_N      = 10    # estudios finales devueltos

# Pesos del score compuesto
W_INCLUSION = 1.0
W_EXCLUSION = 0.5

## 2 — Carga de datos de estudios

Los `.npy` grandes se abren con `mmap` (solo se materializa el subset de candidatos).

In [3]:
clinical_df    = pd.read_parquet(CLINICAL_PARQUET)
inclusion_meta = pd.read_parquet(INCLUSION_META_PATH)
exclusion_meta = pd.read_parquet(EXCLUSION_META_PATH)
condition_meta = pd.read_parquet(CONDITION_META_PATH)

inclusion_emb = np.load(INCLUSION_EMB_PATH, mmap_mode='r')
exclusion_emb = np.load(EXCLUSION_EMB_PATH, mmap_mode='r')
condition_emb = np.load(CONDITION_EMB_PATH)  # chico, en memoria

print(f'Estudios:            {len(clinical_df):,}')
print(f'Criterios inclusion: {len(inclusion_meta):,}  emb {inclusion_emb.shape}')
print(f'Criterios exclusion: {len(exclusion_meta):,}  emb {exclusion_emb.shape}')
print(f'Condiciones:         {len(condition_meta):,}  emb {condition_emb.shape}')

Estudios:            83,962
Criterios inclusion: 750,566  emb (750566, 768)
Criterios exclusion: 678,985  emb (678985, 768)
Condiciones:         83,962  emb (83962, 384)


## 3 — Modelo de embeddings

Misma clase que el notebook 06; `normalize_embeddings=True` hace que dot product = coseno.

In [4]:
class EmbeddingModel:
    """Encapsula SentenceTransformer con encoding normalizado (cosine = dot)."""

    def __init__(self, model_name: str, device: str = 'cuda') -> None:
        self.model = SentenceTransformer(model_name, device=device)

    def encode(self, texts: list[str]) -> np.ndarray:
        """Codifica textos a array float32 normalizado."""
        return self.model.encode(
            texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
        )

## 4 — Formulario del paciente

`Patient` espeja los campos del formulario del frontend (`utils/trial_matcher_1.html`).
`sex` y `country` se guardan pero todavia no se usan (faltan en los datos normalizados).

In [5]:
@dataclass
class Patient:
    """Perfil clinico del paciente (espeja el formulario del frontend)."""
    age: int
    sex: str                                       # 'MALE' | 'FEMALE'  (TODO: falta en datos)
    diagnosis: str                                 # diagnostico principal
    histology: str = ''                            # subtipo histologico
    setting: str = ''                              # estadio / setting de enfermedad
    biomarkers: list[str] = field(default_factory=list)
    ecog: str = ''                                 # performance status
    tx_setting: str = ''                           # intencion / linea de tratamiento
    prior_lines: str = ''                          # lineas de tratamiento previas
    free_text: str = ''                            # detalles clinicos libres
    country: str = ''                              # geo (TODO: falta en datos)

In [6]:
def build_condition_query(patient: Patient) -> str:
    """Texto para el filtro de condicion: diagnostico + histologia + biomarcadores."""
    parts = [patient.diagnosis, patient.histology, *patient.biomarkers]
    return ' '.join(p for p in parts if p)


def build_criteria_query(patient: Patient) -> str:
    """Perfil clinico en lenguaje natural para matchear contra criterios de elegibilidad."""
    lines = [
        f'{patient.age} year old {patient.sex.lower()} patient.',
        f'Diagnosis: {patient.diagnosis}.',
    ]
    if patient.histology:   lines.append(f'Histology: {patient.histology}.')
    if patient.setting:     lines.append(f'Disease setting: {patient.setting}.')
    if patient.biomarkers:  lines.append(f'Biomarkers: {", ".join(patient.biomarkers)}.')
    if patient.ecog:        lines.append(f'ECOG performance status {patient.ecog}.')
    if patient.tx_setting:  lines.append(f'Treatment intent: {patient.tx_setting}.')
    if patient.prior_lines: lines.append(f'Prior therapy lines: {patient.prior_lines}.')
    if patient.free_text:   lines.append(patient.free_text)
    return ' '.join(lines)

## 5 — Filtros duros

Status `RECRUITING` + edad del paciente compatible con `min_age` y el bucket `std_ages`.

In [7]:
def parse_min_age(min_age) -> int:
    """Extrae anios de min_age ('18 Years' -> 18); sin minimo -> 0."""
    if not isinstance(min_age, str):
        return 0
    return int(min_age.split()[0])


def age_bucket(age: int) -> str:
    """Mapea edad al bucket std_ages de ClinicalTrials."""
    if age < 18:
        return 'CHILD'
    if age < 65:
        return 'ADULT'
    return 'OLDER_ADULT'


def filter_studies(clinical_df: pd.DataFrame, patient: Patient) -> set:
    """nct_ids que pasan filtros duros: status RECRUITING + edad compatible."""
    bucket = age_bucket(patient.age)
    eligible = clinical_df[
        (clinical_df['overall_status'] == RECRUITING_STATUS)
        & (clinical_df['min_age'].map(parse_min_age) <= patient.age)
        & (clinical_df['std_ages'].map(lambda b: b is None or bucket in b))
    ]
    return set(eligible['nct_id'])

## 6 — Retrieval semantico

Filtro de condicion -> candidatos; luego score por criterios de inclusion/exclusion.

In [8]:
def condition_candidates(
    query_emb: np.ndarray,
    condition_meta: pd.DataFrame,
    condition_emb: np.ndarray,
    eligible_ids: set,
    top_n: int = CONDITION_TOP_N,
) -> list:
    """Top-N estudios por similitud de condicion, restringido a los elegibles."""
    mask = condition_meta['nct_id'].isin(eligible_ids).to_numpy()
    sims = np.asarray(condition_emb[mask]) @ query_emb
    ids  = condition_meta.loc[mask, 'nct_id'].to_numpy()
    top_idx = np.argsort(-sims)[:top_n]
    return ids[top_idx].tolist()

In [9]:
def score_criteria(
    query_emb: np.ndarray,
    meta: pd.DataFrame,
    emb: np.ndarray,
    candidate_ids: list,
    top_k: int,
) -> dict:
    """Score por estudio = promedio de las top_k similitudes de criterio (0 si no tiene)."""
    mask = meta['nct_id'].isin(set(candidate_ids)).to_numpy()
    sub_ids = meta.loc[mask, 'nct_id'].to_numpy()
    sims = np.asarray(emb[mask]) @ query_emb
    scored = {}
    for nct_id, group in pd.Series(sims, index=sub_ids).groupby(level=0):
        top = np.sort(group.to_numpy())[::-1][:top_k]
        scored[nct_id] = float(top.mean())
    return scored

## 7 — Score compuesto y explicabilidad

In [10]:
def composite_score(
    inclusion_scores: dict,
    exclusion_scores: dict,
    candidate_ids: list,
) -> pd.DataFrame:
    """Combina inclusion (+) y exclusion (-) en un score final por estudio candidato."""
    rows = []
    for nct_id in candidate_ids:
        inc = inclusion_scores.get(nct_id, 0.0)
        exc = exclusion_scores.get(nct_id, 0.0)
        rows.append({
            'nct_id': nct_id,
            'inclusion_score': inc,
            'exclusion_penalty': exc,
            # TODO: + W_RERANK * cross_encoder  + W_GEO * score_geografico
            'final_score': W_INCLUSION * inc - W_EXCLUSION * exc,
        })
    return (
        pd.DataFrame(rows)
        .sort_values('final_score', ascending=False)
        .reset_index(drop=True)
    )


def best_inclusion_match(
    query_emb: np.ndarray,
    meta: pd.DataFrame,
    emb: np.ndarray,
    nct_id: str,
) -> str:
    """Criterio de inclusion mas similar de un estudio (para explicar el match)."""
    mask = (meta['nct_id'] == nct_id).to_numpy()
    texts = meta.loc[mask, 'criterion_text'].to_numpy()
    sims  = np.asarray(emb[mask]) @ query_emb
    return texts[int(np.argmax(sims))]

## 8 — Pipeline online completo

In [11]:
def match_trials(patient: Patient) -> pd.DataFrame:
    """Formulario -> filtros -> retrieval -> score compuesto -> top-N explicado."""
    cond_q = condition_model.encode([build_condition_query(patient)])[0]
    crit_q = criteria_model.encode([build_criteria_query(patient)])[0]

    eligible_ids  = filter_studies(clinical_df, patient)
    candidate_ids = condition_candidates(cond_q, condition_meta, condition_emb, eligible_ids)

    inclusion_scores = score_criteria(crit_q, inclusion_meta, inclusion_emb, candidate_ids, INCLUSION_TOP_K)
    exclusion_scores = score_criteria(crit_q, exclusion_meta, exclusion_emb, candidate_ids, INCLUSION_TOP_K)

    ranked = composite_score(inclusion_scores, exclusion_scores, candidate_ids).head(RESULT_TOP_N)
    ranked['top_inclusion_criterion'] = [
        best_inclusion_match(crit_q, inclusion_meta, inclusion_emb, n) for n in ranked['nct_id']
    ]
    return ranked.merge(
        clinical_df[['nct_id', 'condition_text', 'overall_status']], on='nct_id', how='left'
    )

## 9 — Carga de modelos (una sola vez)

In [12]:
condition_model = EmbeddingModel(CONDITION_MODEL_NAME)
criteria_model  = EmbeddingModel(CRITERIA_MODEL_NAME)

C:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


C:\Users\User\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--pritamdeka--S-PubMedBert-MS-MARCO. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 10 — Ejemplo: paciente de prueba

In [13]:
example_patient = Patient(
    age=58,
    sex='FEMALE',
    diagnosis='Breast Cancer',
    histology='invasive ductal carcinoma',
    setting='metastatic',
    biomarkers=['HER2-positive', 'ER-positive'],
    ecog='1',
    tx_setting='second line',
    prior_lines='1',
    free_text='Prior trastuzumab. Adequate organ function. No active brain metastases.',
)

results = match_trials(example_patient)
print(f'Top {len(results)} estudios para el paciente:')
results[['nct_id', 'final_score', 'inclusion_score', 'exclusion_penalty', 'overall_status']]

Top 10 estudios para el paciente:


,nct_id,final_score,inclusion_score,exclusion_penalty,overall_status
0,NCT07377643,0.942719,0.942719,0.000000,RECRUITING
1,NCT06603597,0.935847,0.935847,0.000000,RECRUITING
2,NCT01785420,0.503869,0.954306,0.900874,RECRUITING
3,NCT05933395,0.498087,0.949637,0.903100,RECRUITING
4,NCT06548529,0.496151,0.927154,0.862006,RECRUITING
5,NCT06435429,0.492354,0.930971,0.877233,RECRUITING
6,NCT06754059,0.491849,0.939731,0.895765,RECRUITING
7,NCT06417801,0.491771,0.955774,0.928006,RECRUITING
8,NCT07294508,0.491370,0.959196,0.935652,RECRUITING
9,NCT06923527,0.491351,0.948930,0.915158,RECRUITING


In [14]:
for _, row in results.iterrows():
    print(f"\n{row['nct_id']}  (score {row['final_score']:.3f})")
    print(f"  Condicion:      {row['condition_text'][:100]}")
    print(f"  Criterio match: {row['top_inclusion_criterion'][:120]}")


NCT07377643  (score 0.943)
  Condicion:      her2-positive breast cancer
  Criterio match: No prior chemotherapy or human epidermal growth factor receptor 2-targeted therapy for unresectable, locally advanced or

NCT06603597  (score 0.936)
  Condicion:      Breast Neoplasms | her2 + breast cancer | breast cancer | registry | her2+ | her2 positive
  Criterio match: Stage I-intravenous human epidermal growth factor receptor 2-positive breast cancer

NCT01785420  (score 0.504)
  Condicion:      Breast Neoplasms | carcinoma breast stage i | her2 positive breast cancer | pre-operative trastuzuma
  Criterio match: a.

1. Female subjects aged 18 years or older.
2. Histologically and/or cytologically confirmed diagnosis of breast canc

NCT05933395  (score 0.498)
  Condicion:      Neoplasm Metastasis | advanced breast cancer | locally recurrent | metastatic | er+ | er+/her2-
  Criterio match: 1. Post-menopausal women ≥18 years of age with metastatic ER+ breast cancer, or with locally recurrent